# Compute Core — VMs, Scale Sets & Availability

Virtual machines are the oldest compute primitive in Azure and still the busiest one. Every higher-level service — App Service, AKS nodes, Databricks clusters, even Azure SQL Managed Instance — runs on a VM somewhere; you just stop seeing it. Understanding how VMs are sized, imaged, priced, given disks and NICs, grouped for availability, and scaled in fleets is the bedrock the rest of the platform sits on.

The shape to hold in your head: a VM is a bundle of compute (size), storage (managed disks), network (NIC), and placement (region + zone + group). Get each piece right and the VM behaves; get one wrong and the symptoms are subtle — performance cliffs, surprise bills, or a quiet single point of failure.

## VM families and sizing

Azure has well over four hundred VM SKUs. They're organised into **families**, each tuned for a different workload profile. The naming convention encodes the family, version, and feature flags:

```
Standard_D8s_v5
         │ │ │
         │ │ └── version (v5 = newer generation)
         │ └──── features (s = premium-SSD-capable; d = local disk; a = AMD; etc.)
         └────── vCPU count (8 cores)
```

The families you will see most:

- **B-series** — burstable, cheap, accumulate CPU credits when idle. Dev/test and low-utilisation workloads. Equivalent to AWS T-series.
- **D-series** — general-purpose balance of vCPU, memory, and temp storage. The default first guess for a web app or API.
- **E-series** — memory-optimised; ~8 GB RAM per vCPU. Caching, in-memory databases, analytics.
- **F-series** — compute-optimised; higher CPU-to-memory ratio. Batch processing, gaming, media encoding.
- **L-series** — storage-optimised, with large NVMe local disks. Cassandra, ClickHouse, anything that wants fast local I/O.
- **M-series** — massive memory (up to several terabytes). SAP HANA, in-memory analytics.
- **N-series** — GPU. NV/NC/ND subfamilies for visualisation, compute, and deep learning respectively.
- **H-series** — high-performance computing with InfiniBand interconnects. Tightly-coupled simulations.

Trailing letters matter. `s` means the SKU can attach premium SSDs. `d` means it has a local temp disk (ephemeral, fast, lost on dealloc). `a` means AMD silicon, often 10–20% cheaper than the Intel equivalent. Versions (`v4`, `v5`, `v6`) bump generation; newer is usually cheaper per unit of work but not always available in every region.

## Images — what the VM boots from

Every VM starts from a disk image. Three sources, in order of how much of the image you control:

- **Marketplace images** — Microsoft, Canonical, Red Hat, third-party ISVs publish OS and pre-configured app images. Versioned. The fastest start.
- **Custom images** — you take a marketplace image, install your runtime, run sysprep / waagent, and capture it. Good for one or two VMs in one region.
- **Azure Compute Gallery** (formerly Shared Image Gallery) — the production way to manage custom images at scale. Holds **image definitions** (the family) and **image versions** (timestamped snapshots), and replicates them across regions and into availability zones. Versioned, signed, and consumable from any subscription in your tenant via RBAC.

The practical pattern for a serious workload: bake a hardened OS image with Packer (or `az image create`), publish it to a Compute Gallery, replicate it to every region you deploy in, and reference it by gallery image version from your VM scale set or template. Patch month means publishing a new version, not editing in place.

AWS equivalent: marketplace images ≈ public AMIs; custom images ≈ private AMIs; Compute Gallery ≈ EC2 Image Builder + cross-region AMI copy.

## Pricing models

The list price you see in the calculator is the **pay-as-you-go (PAYG)** rate. Almost nobody pays it for steady-state workloads — there are at least four ways to cut it.

- **Reserved Instances (RI)** — commit to a specific VM size in a specific region for 1 or 3 years. Up to ~72% off vs PAYG. Pay upfront or monthly. Best for predictable production workloads pinned to a known SKU.
- **Azure Savings Plan for Compute** — commit to an hourly *dollar* spend on compute for 1 or 3 years. Up to ~65% off. More flexible than RIs because the discount applies across VM families and regions automatically. Best when you don't want to bet on a specific SKU.
- **Spot VMs** — bid for unused Azure capacity. Up to ~90% off, but Azure can evict the VM with 30 seconds' notice when it needs the capacity back. Perfect for batch jobs, CI/CD agents, and stateless scale-out workers that can be restarted. Set a max price (or `-1` for the on-demand cap) and an eviction policy (`Deallocate` or `Delete`).
- **Azure Hybrid Benefit** — bring your existing Windows Server or SQL Server licences (with Software Assurance) to Azure VMs. Up to ~40% off Windows VMs, stackable with RIs/Savings Plans. The biggest single-line saving for enterprises migrating Windows workloads.

Combined, a production fleet often pays 80%+ off list: RIs or Savings Plans for the steady-state base, Hybrid Benefit on top, Spot for the burst layer.

## Managed disks

A VM has three classes of disk attached:

- **OS disk** — where the operating system lives. Always managed. One per VM.
- **Data disks** — additional volumes for application data. Up to the VM SKU's documented maximum (typically 4–64).
- **Temp disk** — local SSD on the host. *Ephemeral* — wiped on stop/dealloc. Mounted as `D:` on Windows or `/mnt` on Linux. Use it for swap and scratch, never for data you need to keep.

Managed disks come in five tiers, ordered by latency and price:

| Tier | IOPS / disk | Latency | Use case |
|------|-------------|---------|----------|
| Ultra Disk | up to 160k | sub-ms | high-end OLTP, SAP HANA logs |
| Premium SSD v2 | up to 80k | low single ms | most production workloads |
| Premium SSD | up to 20k | low single ms | classic production |
| Standard SSD | a few hundred | tens of ms | dev/test, low-traffic web |
| Standard HDD | dozens | tens of ms | backup, archival |

**Premium SSD v2** is the newer of the two premium tiers and almost always the better choice — IOPS and throughput are decoupled from disk size, so you can dial them independently. Classic Premium SSD ties IOPS to capacity (a 1 TB disk gets ~5,000 IOPS regardless of whether you need 1k or 10k).

All managed disks are **encrypted at rest by default** with platform-managed keys; flipping to customer-managed keys (CMK) via Key Vault is a checkbox at create time. **Snapshots** are incremental and stored cheaply in standard storage. **Shared disks** allow the same managed disk to attach to multiple VMs simultaneously (Windows Failover Cluster, Linux pacemaker).

## NICs and accelerated networking

Every VM has at least one **network interface card (NIC)** attached to a subnet in a virtual network. Larger SKUs allow multiple NICs — useful for separating management and data planes, or for network virtual appliances.

**Accelerated Networking** is a single toggle that moves the network datapath off the host CPU and onto the SmartNIC hardware (SR-IOV under the hood). The effect on most workloads:

- Throughput jumps from a few Gbps to 30+ Gbps on supported SKUs.
- Latency drops noticeably.
- The host CPU stops burning cycles on packet processing.

It is free, supported on almost every modern SKU, and there is essentially no reason *not* to turn it on. Many older templates and Marketplace images still ship with it off by default — check before deploying.

## Proximity Placement Groups

Within a region, your VMs can land in different datacenters separated by enough fibre to add a few milliseconds of latency. For chatty multi-tier workloads — a web tier talking to a cache talking to a database — that overhead adds up.

A **Proximity Placement Group (PPG)** is a hint to the Azure placer: "keep these VMs physically close to each other." The first VM in the PPG pins the datacenter; later VMs are placed in the same one when capacity allows.

The trade-off is real: tighter co-location means more allocation failures when the chosen datacenter is full, and it conflicts with availability zone spread. PPGs solve a real latency problem but are not a default — use them only when you have measured the latency tax and decided it matters.

## Availability — Sets vs Zones vs Regions

Azure offers three nested levels of redundancy for VMs. They answer different failure questions, and they cost different amounts.

```
Single VM            ── one rack, one PSU                  (no HA)
Availability Set     ── multiple racks in one DC           (rack/PSU failure)
Availability Zones   ── multiple DCs in one region         (whole-DC failure)
Multi-region         ── DR copy in a paired/distant region (whole-region failure)
```

**Availability Set (AS)** — the original primitive. VMs in an AS are spread across **fault domains** (racks with independent power and network) and **update domains** (groups Azure patches at different times). Defaults: 2–3 fault domains, 5 update domains. The promise: a planned maintenance event or a single-rack failure takes down at most one update domain or one fault domain. SLA: 99.95% for VMs in an AS.

**Availability Zones (AZ)** — physically separate datacenters within a region. Spread a VM scale set across all three zones and the SLA jumps to 99.99%. AZ is the newer, stronger primitive and the default recommendation for any production workload in a zone-enabled region.

**Multi-region** — the disaster recovery tier. Replicate state to a paired or distant region; fail over if the primary region is unavailable. Cost roughly doubles; complexity climbs sharply.

Rules of thumb: start with AZ. Use AS only in regions without zones or when AZ pricing is prohibitive. Add multi-region only for tiers where a regional outage is a business-critical event. **AS and AZ are mutually exclusive** for a given VM — you pick one. (VM Scale Sets can span zones; classic single VMs sit in one zone if zonal at all.)

In [ ]:
# Provision a VM end-to-end with az.

RG=rg-compute-demo
az group create --name $RG --location eastus

# 1. Create a VM in zone 1 with accelerated networking and a premium OS disk.
az vm create \
  --resource-group $RG \
  --name vm-web-01 \
  --image Ubuntu2204 \
  --size Standard_D4s_v5 \
  --zone 1 \
  --os-disk-name vm-web-01-os \
  --storage-sku Premium_LRS \
  --accelerated-networking true \
  --admin-username azureuser \
  --generate-ssh-keys

# 2. Attach a 256 GB Premium SSD v2 data disk.
az vm disk attach \
  --resource-group $RG \
  --vm-name vm-web-01 \
  --name vm-web-01-data \
  --new --size-gb 256 --sku PremiumV2_LRS

# 3. Spot variant — same VM, 80% cheaper, can be evicted.
az vm create \
  --resource-group $RG \
  --name vm-batch-spot-01 \
  --image Ubuntu2204 \
  --size Standard_D4s_v5 \
  --priority Spot \
  --eviction-policy Deallocate \
  --max-price -1 \
  --admin-username azureuser \
  --generate-ssh-keys

# 4. Tear it all down.
az group delete --name $RG --yes --no-wait

## VM Scale Sets — fleets, not pets

A **Virtual Machine Scale Set (VMSS)** is a managed group of identical VMs that scale together. Behind the curtain it is a template (the *model*) plus N instances; you change the model once and VMSS reconciles every instance toward it.

Two orchestration modes, and the choice is permanent:

- **Uniform orchestration** — the classic mode. All VMs are identical, fast to create, optimised for thousands of instances of the same shape. You give up some flexibility (no mixing SKUs, no per-instance tweaks).
- **Flexible orchestration** — the newer default. Each VM is a first-class resource you can see in the resource group; you can mix sizes, pin some to zones explicitly, and attach VMSS-style autoscale only when you want it. Slightly slower create per instance, but the operational model is far easier to reason about. **Use Flexible for new scale sets** unless a specific feature forces Uniform.

Both modes spread instances across **availability zones** when you ask, place them in **fault domains** for rack-level redundancy, and surface a single resource (the scale set) to RBAC and tagging.

AWS equivalent: VMSS ≈ EC2 Auto Scaling Group. The shape is similar; VMSS Uniform feels closer to legacy ASG with launch configurations, while Flexible feels closer to modern ASGs with launch templates.

## Autoscale, instance protection, rolling upgrades

VMSS scales horizontally on rules you write against **metrics** (CPU, memory, queue depth, custom Application Insights metric) or **schedules** (08:00 weekdays = 20 instances, weekends = 4). Rules have an aggregation window, a threshold, and a cooldown — without cooldowns you get scaling oscillation, and the most common autoscale bug is a too-aggressive threshold that toggles between scale-in and scale-out.

Two operational levers worth knowing:

- **Instance protection** — flag an instance as protected against scale-in (so an in-flight job is not killed) or against the scale-set model (so a one-off manual tweak isn't reverted on next reconcile). Useful, but if you find yourself reaching for it often, your model is wrong.
- **Rolling upgrades** — when you update the model (a new image version, a different SKU), VMSS upgrades instances in batches. You choose a batch size (`maxBatchInstancePercent`), a healthy-instance threshold, and pause-between-batches. Combined with **application health extension** probes or a Load Balancer health check, a bad image fails the upgrade automatically rather than rolling out everywhere.

**Ephemeral OS disks** are a VMSS trick worth knowing: instead of a managed disk, the OS lives on the VM's local SSD. Zero storage cost, much faster reimage on update, but you cannot stop-deallocate (the disk would vanish). Perfect for stateless web tiers.

In [ ]:
# A VM Scale Set in Flexible orchestration, spread across 3 zones, with autoscale.

RG=rg-vmss-demo
az group create --name $RG --location eastus

# 1. Create the scale set.
az vmss create \
  --resource-group $RG \
  --name vmss-web \
  --orchestration-mode Flexible \
  --image Ubuntu2204 \
  --vm-sku Standard_D2s_v5 \
  --instance-count 3 \
  --zones 1 2 3 \
  --upgrade-policy-mode Rolling \
  --admin-username azureuser \
  --generate-ssh-keys

# 2. Add an autoscale profile: 2–20 instances, scale on CPU.
az monitor autoscale create \
  --resource-group $RG \
  --resource vmss-web \
  --resource-type Microsoft.Compute/virtualMachineScaleSets \
  --name autoscale-web \
  --min-count 2 --max-count 20 --count 3

az monitor autoscale rule create \
  --resource-group $RG --autoscale-name autoscale-web \
  --condition "Percentage CPU > 70 avg 5m" --scale out 2

az monitor autoscale rule create \
  --resource-group $RG --autoscale-name autoscale-web \
  --condition "Percentage CPU < 30 avg 10m" --scale in 1

## Azure Dedicated Host

Most VMs land on shared hardware — you and other tenants split the same physical host. For regulated or licence-sensitive workloads, **Azure Dedicated Host** gives you the whole physical server. You pay for the host by the hour and run any number of VMs on it; you choose which generation of host (Dsv3-Type1, Esv5-Type2, etc.) and which datacenter zone.

Two things Dedicated Host buys you:

- **Physical isolation** for compliance regimes (PCI, FedRAMP High, some financial regulators) that require it.
- **Bring-your-own-licence economics** for software licensed per-physical-core — Windows Server, SQL Server, some Oracle products. Saving the per-core licence on every VM that runs on the host can dwarf the host rental cost.

If neither applies, stay on the shared fleet — Dedicated Host is meaningfully more expensive per vCPU and harder to scale.

## Putting it together

A production VM workload, top to bottom:

1. **Image** — hardened OS published as a Compute Gallery version, replicated to your regions.
2. **SKU + pricing** — D-series for general-purpose, sized from real load data; Reserved or Savings Plan for the steady-state base, Spot for burst, Hybrid Benefit if you bring Windows licences.
3. **Disks** — Premium SSD v2 for OS and data; Ultra only if a benchmark proves you need it; encryption with CMK if compliance demands.
4. **Networking** — accelerated networking on; NIC in the subnet with the right NSG; no public IP unless an explicit gateway/LB demands.
5. **Placement** — VMSS Flexible, spread across all three availability zones in the region; multi-region only if RTO/RPO requires.
6. **Scale** — autoscale on CPU + queue depth, with cooldowns; rolling upgrades with health probes; ephemeral OS disks for stateless tiers.

Get those six right and the VM tier disappears as a problem to think about, which is exactly what you want — your attention belongs higher in the stack, on the app and the data.